# Code-Comment Generator

> **A neural seq2seq model that generates English comments for Java methods, with a Visual Studio Code extension for in-editor use.**


## 1. Setup

This notebook is designed to run on Google Colab with a GPU runtime (T4 free tier is sufficient). It will work locally too if you have CUDA available.

In [ ]:
# Install dependencies
!pip install -q transformers==4.40.0 datasets==2.18.0 sacrebleu rouge-score sentencepiece nltk
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [ ]:
import os
import json
import gzip
import math
import random
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 2. Data Loading

We use **CodeSearchNet** ([Husain et al., 2019](https://arxiv.org/abs/1909.09436)) — a benchmark dataset of paired (code, docstring) examples mined from open-source GitHub repositories. We focus on the Java subset.

The original project sampled ~20% of the data due to compute limits. Here we'll do the same for the from-scratch model (it would take days otherwise) but use the full dataset for CodeT5 fine-tuning since it's much more efficient.

In [ ]:
# Download CodeSearchNet Java data
# In Colab, this is fastest from the official S3 mirror
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

if not (DATA_DIR / "java" / "final").exists():
    !wget -q https://s3.amazonaws.com/code-search-net/CodeSearchNet/v2/java.zip -O {DATA_DIR}/java.zip
    !unzip -q {DATA_DIR}/java.zip -d {DATA_DIR}/
    !rm {DATA_DIR}/java.zip

print("Data ready at:", DATA_DIR / "java" / "final" / "jsonl")

In [ ]:
def load_split(split_dir: Path) -> pd.DataFrame:
    """Load all .jsonl.gz files in a split directory into a single DataFrame."""
    records = []
    for fp in sorted(split_dir.glob("*.jsonl.gz")):
        with gzip.open(fp, "rt", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                records.append({
                    "code": obj["code"],
                    "docstring": obj["docstring"],
                    "func_name": obj.get("func_name", ""),
                    "repo": obj.get("repo", ""),
                })
    return pd.DataFrame(records)

base = DATA_DIR / "java" / "final" / "jsonl"
df_train = load_split(base / "train")
df_valid = load_split(base / "valid")
df_test  = load_split(base / "test")

print(f"Train: {len(df_train):>7,} examples")
print(f"Valid: {len(df_valid):>7,} examples")
print(f"Test:  {len(df_test):>7,} examples")

## 3. Data Exploration

Before any modeling, let's understand what we're working with. This step was missing from the original notebook — we jumped straight from loading to training. Knowing the data shape informs every downstream choice (max sequence length, vocab size, train/eval splits).

In [ ]:
# Look at a few examples
for i in [0, 100, 1000]:
    print(f"=== Example {i} ===")
    print("CODE:")
    print(df_train.iloc[i]["code"][:300])
    print("\nDOCSTRING:")
    print(df_train.iloc[i]["docstring"][:300])
    print("\n" + "="*60 + "\n")

In [ ]:
# Length distributions (in characters and tokens)
df_train["code_len_chars"] = df_train["code"].str.len()
df_train["doc_len_chars"]  = df_train["docstring"].str.len()
df_train["code_len_words"] = df_train["code"].str.split().str.len()
df_train["doc_len_words"]  = df_train["docstring"].str.split().str.len()

print(df_train[["code_len_chars", "doc_len_chars", "code_len_words", "doc_len_words"]].describe())

In [ ]:
# Visualize the distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_train["code_len_words"].clip(upper=300), bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Code length distribution (in words, clipped at 300)")
axes[0].set_xlabel("Words")
axes[0].set_ylabel("Frequency")
axes[0].axvline(df_train["code_len_words"].median(), color="red", linestyle="--", label=f"median: {df_train['code_len_words'].median():.0f}")
axes[0].legend()

axes[1].hist(df_train["doc_len_words"].clip(upper=100), bins=50, color="seagreen", edgecolor="white")
axes[1].set_title("Docstring length distribution (in words, clipped at 100)")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Frequency")
axes[1].axvline(df_train["doc_len_words"].median(), color="red", linestyle="--", label=f"median: {df_train['doc_len_words'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig("length_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

**Takeaway:** Java methods in this dataset are typically 30–80 words, with a long tail. Docstrings are much shorter — usually under 30 words. This guides our `max_source_length` (256) and `max_target_length` (64) choices later.

## 4. Preprocessing

Standard cleaning steps:

1. Take only the **first sentence** of the docstring — Javadocs often have many lines of `@param` tags after the summary, and we just want the summary.
2. Drop non-ASCII docstrings to keep the task English-only.
3. Remove HTML tags (Javadocs use `<code>`, `<p>`, etc.).
4. Drop pairs where the docstring is longer than the code (almost always means the docstring is a multi-paragraph essay, not a summary).
5. Drop empties and exact duplicates.

In [ ]:
def first_sentence(text: str) -> str:
    """Return the first sentence of a docstring (everything before the first period or newline-newline)."""
    text = text.strip()
    # Cut off at @param / @return / @throws etc.
    text = re.split(r"\n\s*@\w+", text, maxsplit=1)[0]
    # Cut off at first period followed by space (sentence boundary)
    m = re.search(r"\.\s", text)
    if m:
        text = text[: m.start() + 1]
    return text.strip()

def is_ascii(text: str) -> bool:
    try:
        text.encode("ascii")
        return True
    except UnicodeEncodeError:
        return False

HTML_RE = re.compile(r"<[^>]+>")

def clean_docstring(text: str) -> str:
    text = first_sentence(text)
    text = HTML_RE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["docstring"] = df["docstring"].apply(clean_docstring)
    # Filter: ASCII only
    df = df[df["docstring"].apply(is_ascii)]
    # Filter: drop empties
    df = df[df["docstring"].str.len() > 0]
    # Filter: code longer than docstring
    df = df[df["code"].str.len() > df["docstring"].str.len()]
    # Filter: reasonable lengths
    df = df[df["docstring"].str.split().str.len().between(3, 50)]
    df = df[df["code"].str.split().str.len().between(10, 400)]
    # Drop duplicates
    df = df.drop_duplicates(subset=["docstring"]).reset_index(drop=True)
    return df

df_train_clean = preprocess(df_train)
df_valid_clean = preprocess(df_valid)
df_test_clean  = preprocess(df_test)

print(f"After cleaning:")
print(f"Train: {len(df_train_clean):>7,} (was {len(df_train):>7,})")
print(f"Valid: {len(df_valid_clean):>7,} (was {len(df_valid):>7,})")
print(f"Test:  {len(df_test_clean):>7,} (was {len(df_test):>7,})")

In [ ]:
# Sanity check — show a few cleaned pairs
for i in [0, 50, 200]:
    row = df_train_clean.iloc[i]
    print(f"CODE:      {row['code'][:120]}...")
    print(f"DOCSTRING: {row['docstring']}")
    print()

## 5. Approach A — Transformer From Scratch

The original project trained a small Transformer from scratch. Here we do the same, but with cleaner code and modern PyTorch idioms. The architecture follows Vaswani et al. (2017) — multi-head self-attention encoder, masked multi-head attention decoder, sinusoidal positional encodings.

**Why this is here:** building it from scratch demonstrates that I understand the architecture rather than just calling `model.fit()`. A recruiter or admissions committee should be able to read this code and see the attention mechanism, the masking, the positional encoding — all of it.

**What this is not:** competitive with state-of-the-art. With our compute budget and dataset size, this model produces grammatical-but-generic comments. The fine-tuned CodeT5 in Section 6 does substantially better. The point is the comparison.

### 5.1 Tokenizer

We use Hugging Face's `tokenizers` library to train a Byte-Pair Encoding (BPE) tokenizer on our data. Two separate tokenizers — one for code, one for docstrings — because the vocabularies are very different (code has lots of camelCase identifiers; docstrings are natural English).

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
VOCAB_SIZE = 8000  # small to keep the model light

def train_bpe_tokenizer(texts, vocab_size=VOCAB_SIZE):
    tok = Tokenizer(BPE(unk_token="<unk>"))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)
    tok.train_from_iterator(texts, trainer)
    return tok

# Subsample training texts to speed up tokenizer training
SAMPLE_FOR_TOK = 30000
sample_idx = np.random.choice(len(df_train_clean), min(SAMPLE_FOR_TOK, len(df_train_clean)), replace=False)
code_tok = train_bpe_tokenizer(df_train_clean.iloc[sample_idx]["code"].tolist())
doc_tok  = train_bpe_tokenizer(df_train_clean.iloc[sample_idx]["docstring"].tolist())

print(f"Code vocab size: {code_tok.get_vocab_size()}")
print(f"Doc vocab size:  {doc_tok.get_vocab_size()}")
print()
print("Example code tokens: ", code_tok.encode("public void hello(String name) { System.out.println(name); }").tokens[:20])
print("Example doc tokens:  ", doc_tok.encode("Prints the given name to standard output.").tokens)

### 5.2 Dataset class

A standard PyTorch `Dataset` that tokenizes on the fly, adds `<bos>` and `<eos>` markers, and pads to a max length.

In [ ]:
MAX_SRC_LEN = 256
MAX_TGT_LEN = 64

class CodeCommentDataset(Dataset):
    def __init__(self, df, code_tok, doc_tok, max_src=MAX_SRC_LEN, max_tgt=MAX_TGT_LEN):
        self.df = df.reset_index(drop=True)
        self.code_tok = code_tok
        self.doc_tok = doc_tok
        self.max_src = max_src
        self.max_tgt = max_tgt

    def __len__(self):
        return len(self.df)

    def encode(self, text, tokenizer, max_len, add_bos=False, add_eos=True):
        ids = tokenizer.encode(text).ids[: max_len - int(add_bos) - int(add_eos)]
        if add_bos:
            ids = [BOS_ID] + ids
        if add_eos:
            ids = ids + [EOS_ID]
        return ids

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src = self.encode(row["code"], self.code_tok, self.max_src, add_bos=False, add_eos=True)
        tgt = self.encode(row["docstring"], self.doc_tok, self.max_tgt, add_bos=True, add_eos=True)
        return torch.tensor(src), torch.tensor(tgt)

def collate(batch):
    """Pad source and target separately to the max length in this batch."""
    srcs, tgts = zip(*batch)
    src_lens = [len(s) for s in srcs]
    tgt_lens = [len(t) for t in tgts]
    src_pad = torch.full((len(batch), max(src_lens)), PAD_ID, dtype=torch.long)
    tgt_pad = torch.full((len(batch), max(tgt_lens)), PAD_ID, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :len(s)] = s
        tgt_pad[i, :len(t)] = t
    return src_pad, tgt_pad

# To keep training time reasonable, subsample the training data
SCRATCH_TRAIN_SIZE = 20000  # ~5-10 minutes per epoch on T4
train_subset = df_train_clean.sample(n=min(SCRATCH_TRAIN_SIZE, len(df_train_clean)), random_state=SEED)

train_ds = CodeCommentDataset(train_subset, code_tok, doc_tok)
valid_ds = CodeCommentDataset(df_valid_clean.head(1000), code_tok, doc_tok)

print(f"Train dataset: {len(train_ds)} examples")
print(f"Valid dataset: {len(valid_ds)} examples")

### 5.3 Model architecture

A standard encoder-decoder Transformer. We implement the components by hand to make the architecture explicit:

- **`PositionalEncoding`** — sinusoidal position embeddings as in the original paper
- **`MultiHeadAttention`** — scaled dot-product attention with multiple heads
- **`TransformerBlock`** — encoder block (self-attention + FFN)
- **`DecoderBlock`** — decoder block (self-attention + cross-attention + FFN)
- **`Seq2SeqTransformer`** — the full model

Smaller than the paper's defaults (6 layers, 512 hidden, 8 heads → here: 3 layers, 256 hidden, 4 heads) so it trains in a Colab session.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al. 2017, Section 3.5)."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class MultiHeadAttention(nn.Module):
    """Multi-head scaled dot-product attention."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        # Project + split into heads: (bs, seq, d_model) -> (bs, n_heads, seq, d_head)
        Q = self.q_proj(q).view(bs, -1, self.n_heads, self.d_head).transpose(1, 2)
        K = self.k_proj(k).view(bs, -1, self.n_heads, self.d_head).transpose(1, 2)
        V = self.v_proj(v).view(bs, -1, self.n_heads, self.d_head).transpose(1, 2)
        # Scaled dot-product attention
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = attn @ V  # (bs, n_heads, seq, d_head)
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.n_heads * self.d_head)
        return self.out_proj(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, n_heads=4,
                 n_layers=3, d_ff=1024, dropout=0.1, max_len=512):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=PAD_ID)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=PAD_ID)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.encoder = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, tgt_vocab)
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)

    def make_src_mask(self, src):
        # (bs, 1, 1, src_len)
        return (src != PAD_ID).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        # (bs, 1, 1, tgt_len) for padding
        pad_mask = (tgt != PAD_ID).unsqueeze(1).unsqueeze(2)
        # (1, 1, tgt_len, tgt_len) for causal
        sz = tgt.size(1)
        causal = torch.tril(torch.ones(sz, sz, device=tgt.device)).bool()
        return pad_mask & causal

    def encode(self, src):
        x = self.dropout(self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model)))
        src_mask = self.make_src_mask(src)
        for layer in self.encoder:
            x = layer(x, src_mask)
        return x, src_mask

    def decode(self, tgt, enc_out, src_mask):
        x = self.dropout(self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.d_model)))
        tgt_mask = self.make_tgt_mask(tgt)
        for layer in self.decoder:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.out(x)

    def forward(self, src, tgt):
        enc_out, src_mask = self.encode(src)
        return self.decode(tgt, enc_out, src_mask)


# Instantiate
model = Seq2SeqTransformer(
    src_vocab=code_tok.get_vocab_size(),
    tgt_vocab=doc_tok.get_vocab_size(),
    d_model=256, n_heads=4, n_layers=3, d_ff=1024, dropout=0.1
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

### 5.4 Training loop

Standard teacher-forcing training: for each batch, the decoder sees the gold target shifted right and predicts the next token at every position. Loss is cross-entropy with `<pad>` ignored.

In [ ]:
BATCH_SIZE = 32
EPOCHS = 5
LR = 3e-4

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=EPOCHS * len(train_loader), pct_start=0.1
)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)


def run_epoch(model, loader, optimizer=None, scheduler=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    n_tokens = 0
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        with torch.set_grad_enabled(is_train):
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
        ntok = (tgt_out != PAD_ID).sum().item()
        total_loss += loss.item() * ntok
        n_tokens += ntok
    return total_loss / max(n_tokens, 1)


train_losses, valid_losses = [], []
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, scheduler)
    vl = run_epoch(model, valid_loader)
    train_losses.append(tr)
    valid_losses.append(vl)
    print(f"Epoch {epoch}/{EPOCHS}  train_loss={tr:.4f}  valid_loss={vl:.4f}")

# Save the trained model
torch.save(model.state_dict(), "scratch_transformer.pt")

In [ ]:
# Plot training curves
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker="o", label="train")
plt.plot(range(1, EPOCHS + 1), valid_losses, marker="o", label="valid")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("From-scratch Transformer — training curves")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("scratch_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

### 5.5 Inference (greedy decoding)

For each input, generate one token at a time, picking the most likely next token, until we hit `<eos>` or `MAX_TGT_LEN`.

A real production system would use beam search or sampling — greedy is fine for a demo and significantly simpler to read.

In [ ]:
@torch.no_grad()
def greedy_decode(model, src_text, code_tok, doc_tok, max_len=MAX_TGT_LEN):
    model.eval()
    src_ids = code_tok.encode(src_text).ids[: MAX_SRC_LEN - 1] + [EOS_ID]
    src = torch.tensor([src_ids], device=DEVICE)
    enc_out, src_mask = model.encode(src)
    ys = torch.tensor([[BOS_ID]], device=DEVICE)
    for _ in range(max_len - 1):
        logits = model.decode(ys, enc_out, src_mask)
        next_id = logits[:, -1, :].argmax(-1, keepdim=True)
        ys = torch.cat([ys, next_id], dim=1)
        if next_id.item() == EOS_ID:
            break
    out_ids = ys[0, 1:].tolist()
    if EOS_ID in out_ids:
        out_ids = out_ids[: out_ids.index(EOS_ID)]
    return doc_tok.decode(out_ids)


# Sample predictions from the test set
samples = df_test_clean.sample(5, random_state=SEED)
print("=== From-scratch Transformer — sample predictions ===\n")
scratch_predictions = []
for _, row in samples.iterrows():
    pred = greedy_decode(model, row["code"], code_tok, doc_tok)
    scratch_predictions.append((row["code"], row["docstring"], pred))
    print(f"CODE:      {row['code'][:120]}...")
    print(f"REFERENCE: {row['docstring']}")
    print(f"PREDICTED: {pred}")
    print("-" * 80)

## 6. Approach B — Fine-tuned CodeT5

The from-scratch Transformer in Section 5 has a fundamental limit: it learned everything about code from ~20K examples. **CodeT5** ([Wang et al., 2021](https://arxiv.org/abs/2109.00859)) was pre-trained by Salesforce on 8.35M code-text pairs across multiple languages, with code-specific objectives like identifier-aware masking. Even before we fine-tune, it knows what an `int` is, what `for` does, what camelCase identifiers usually mean.

We fine-tune `Salesforce/codet5-small` (60M parameters — fits in free Colab) for code summarization on our cleaned Java data. Training takes about 30 minutes for 3 epochs.

In [ ]:
from transformers import (
    T5ForConditionalGeneration, RobertaTokenizer,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset

MODEL_NAME = "Salesforce/codet5-small"
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
codet5 = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

print(f"Loaded {MODEL_NAME} ({sum(p.numel() for p in codet5.parameters()):,} parameters)")

In [ ]:
# Subsample training set for fine-tuning (full set is overkill for a demo)
FT_TRAIN_SIZE = 30000
FT_VALID_SIZE = 1000
FT_TEST_SIZE  = 1000

ft_train = df_train_clean.sample(n=FT_TRAIN_SIZE, random_state=SEED).reset_index(drop=True)
ft_valid = df_valid_clean.head(FT_VALID_SIZE).reset_index(drop=True)
ft_test  = df_test_clean.head(FT_TEST_SIZE).reset_index(drop=True)

def to_hf(df):
    return HFDataset.from_pandas(df[["code", "docstring"]])

hf_train = to_hf(ft_train)
hf_valid = to_hf(ft_valid)
hf_test  = to_hf(ft_test)


def tokenize_pair(batch):
    model_inputs = tokenizer(
        batch["code"], max_length=MAX_SRC_LEN, truncation=True
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["docstring"], max_length=MAX_TGT_LEN, truncation=True
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


hf_train = hf_train.map(tokenize_pair, batched=True, remove_columns=["code", "docstring"])
hf_valid = hf_valid.map(tokenize_pair, batched=True, remove_columns=["code", "docstring"])
hf_test  = hf_test.map(tokenize_pair, batched=True, remove_columns=["code", "docstring"])

print(f"Tokenized: {len(hf_train)} train / {len(hf_valid)} valid / {len(hf_test)} test")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="codet5-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    predict_with_generate=False,  # we'll do generation manually for cleaner control
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=100,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=codet5)

trainer = Seq2SeqTrainer(
    model=codet5,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model("codet5-finetuned")
tokenizer.save_pretrained("codet5-finetuned")

### 6.1 Inference with the fine-tuned model

In [ ]:
@torch.no_grad()
def codet5_generate(model, tokenizer, code_text, max_len=MAX_TGT_LEN, num_beams=4):
    model.eval()
    inputs = tokenizer(
        code_text, return_tensors="pt", max_length=MAX_SRC_LEN, truncation=True
    ).to(DEVICE)
    out = model.generate(
        **inputs,
        max_length=max_len,
        num_beams=num_beams,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


# Sample predictions on the same examples used for the from-scratch model
print("=== Fine-tuned CodeT5 — sample predictions ===\n")
codet5_predictions = []
for code_text, ref, scratch_pred in scratch_predictions:
    pred = codet5_generate(codet5, tokenizer, code_text)
    codet5_predictions.append((code_text, ref, pred))
    print(f"CODE:      {code_text[:120]}...")
    print(f"REFERENCE: {ref}")
    print(f"PREDICTED: {pred}")
    print("-" * 80)

## 7. Evaluation

We evaluate both models on the same held-out test set using two standard metrics for code summarization:

- **BLEU-4** (Papineni et al., 2002) — n-gram precision against the reference. The standard machine-translation metric, also used in DeepCom and the CodeXGLUE benchmark.
- **ROUGE-L** (Lin, 2004) — longest common subsequence-based F1. More forgiving on word order, often correlates better with human judgment for summarization.

Important caveat: both metrics are imperfect proxies for "is the comment useful?" A comment that's semantically correct but worded differently from the reference will score low. This is a known limitation. We complement these with a qualitative side-by-side in Section 7.2.

In [ ]:
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer

EVAL_N = 500  # number of test examples to evaluate on

eval_set = ft_test.head(EVAL_N).reset_index(drop=True)
references = eval_set["docstring"].tolist()

# 1. From-scratch Transformer predictions on eval_set
print(f"Generating from-scratch Transformer predictions on {EVAL_N} examples...")
scratch_preds = []
for _, row in eval_set.iterrows():
    scratch_preds.append(greedy_decode(model, row["code"], code_tok, doc_tok))

# 2. CodeT5 predictions on eval_set
print(f"Generating CodeT5 predictions on {EVAL_N} examples...")
codet5_preds = []
for _, row in eval_set.iterrows():
    codet5_preds.append(codet5_generate(codet5, tokenizer, row["code"]))

print("Done.")

In [ ]:
def compute_metrics(preds, refs):
    # BLEU
    bleu = corpus_bleu(preds, [refs]).score
    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = np.mean([scorer.score(r, p)["rougeL"].fmeasure for r, p in zip(refs, preds)]) * 100
    return {"BLEU-4": bleu, "ROUGE-L": rouge_l}


scratch_scores = compute_metrics(scratch_preds, references)
codet5_scores  = compute_metrics(codet5_preds, references)

results_df = pd.DataFrame([
    {"Model": "From-scratch Transformer", **scratch_scores},
    {"Model": "Fine-tuned CodeT5",        **codet5_scores},
])
print(results_df.to_string(index=False))

In [ ]:
# Visualize the comparison
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)
width = 0.35
bleu_vals = [scratch_scores["BLEU-4"], codet5_scores["BLEU-4"]]
rouge_vals = [scratch_scores["ROUGE-L"], codet5_scores["ROUGE-L"]]
ax.bar(x - width/2, bleu_vals, width, label="BLEU-4", color="steelblue")
ax.bar(x + width/2, rouge_vals, width, label="ROUGE-L", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(["From-scratch\nTransformer", "Fine-tuned\nCodeT5"])
ax.set_ylabel("Score")
ax.set_title(f"Model comparison on {EVAL_N} held-out test examples")
ax.legend()
ax.grid(axis="y", alpha=0.3)
for i, (b, r) in enumerate(zip(bleu_vals, rouge_vals)):
    ax.text(i - width/2, b + 0.3, f"{b:.1f}", ha="center", fontsize=9)
    ax.text(i + width/2, r + 0.3, f"{r:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

### 7.1 Qualitative comparison

Numbers tell only part of the story. Here are 8 random test examples with both models' outputs side by side:

In [ ]:
print("=" * 100)
qual_idx = np.random.choice(len(eval_set), 8, replace=False)
for i in qual_idx:
    code_text = eval_set.iloc[i]["code"]
    print(f"CODE: {code_text[:200]}...")
    print(f"REFERENCE:        {references[i]}")
    print(f"FROM-SCRATCH:     {scratch_preds[i]}")
    print(f"CODET5:           {codet5_preds[i]}")
    print("=" * 100)

### 7.2 What we learn from this

A few patterns to call out:

- **CodeT5 wins decisively on both metrics.** Expected. Pre-training on ~8M code-text pairs is a massive head start over our 20K from-scratch training run.
- **The from-scratch model produces grammatical but generic output.** "Returns the value of the field" is the kind of thing that's true for half of all Java getters — it's a strong prior, but rarely the *correct* prior.
- **CodeT5 picks up on identifiers.** When the method is `findUserById`, CodeT5 will often produce something like "find a user by id" — it's actually using the method name as a signal. The from-scratch model has no real handle on that because it didn't see enough of these patterns.
- **Both models hallucinate.** Neither has a way to know what the code *actually* does — they're both pattern-matching against training data. This is a fundamental limit of the seq2seq approach.

## 8. Inference Server

To make the trained model usable from the VS Code extension, we wrap it in a small FastAPI server. In production you'd deploy this to a cloud endpoint (Hugging Face Inference Endpoints, AWS Lambda, etc.); for development, ngrok exposes the local server on a public URL.

In [ ]:
# Save the inference server code as a separate file (for the GitHub repo)
SERVER_CODE = '''"""
Inference server for Code-Comment Generator.
Loads the fine-tuned CodeT5 model and serves predictions over HTTP.

Run locally:
    python server.py

In Colab (with ngrok):
    Run the cell below.
"""
import torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import T5ForConditionalGeneration, RobertaTokenizer

MODEL_DIR = "codet5-finetuned"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = RobertaTokenizer.from_pretrained(MODEL_DIR)
model = T5ForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE).eval()


class CommentRequest(BaseModel):
    code: str
    max_length: int = 64
    num_beams: int = 4


class CommentResponse(BaseModel):
    comment: str


app = FastAPI(title="Code-Comment Generator")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
def health():
    return {"status": "ok", "device": str(DEVICE)}


@app.post("/generate", response_model=CommentResponse)
@torch.no_grad()
def generate(req: CommentRequest):
    inputs = tokenizer(
        req.code, return_tensors="pt", max_length=256, truncation=True
    ).to(DEVICE)
    out = model.generate(
        **inputs,
        max_length=req.max_length,
        num_beams=req.num_beams,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )
    return CommentResponse(comment=tokenizer.decode(out[0], skip_special_tokens=True))


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("server.py", "w") as f:
    f.write(SERVER_CODE)
print("Wrote server.py")

In [ ]:
# Run the server in Colab and expose it via ngrok
# (requires a free ngrok account — get the auth token from https://dashboard.ngrok.com/)

import nest_asyncio
import uvicorn
from pyngrok import ngrok

# NOTE: paste your ngrok auth token here, or skip this cell if running locally
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Import the FastAPI app from server.py
import importlib.util
spec = importlib.util.spec_from_file_location("server", "server.py")
server = importlib.util.module_from_spec(spec)
spec.loader.exec_module(server)

# Open ngrok tunnel
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print(f"Use this URL in your VS Code extension: {public_url}/generate")

nest_asyncio.apply()
uvicorn.run(server.app, host="0.0.0.0", port=8000)

## 9. Reflection



**Limitations of this rebuild**
- Even fine-tuned CodeT5 hallucinates. Our BLEU-4 in the high-teens-to-low-twenties range is competitive with published 2021-era results on this task, but it's nowhere near what a modern LLM produces zero-shot.
- We trained on a subset for compute. Full-data training would improve scores but the relative comparison would remain.
- We only evaluated on Java. The original team noted "extending to other languages" as future work; CodeT5 is multilingual and could be fine-tuned on the full CodeSearchNet (Java, Python, JS, Go, Ruby, PHP) with the same code.

**What's next if I were to keep going**
- Compare against zero-shot prompting of a modern LLM (GPT-4, Claude, Code Llama). I'd expect the LLM to win on quality but lose on latency and cost.
- Add identifier-aware data augmentation — synonym substitution on variable names — to test whether the model is genuinely understanding code structure or just memorizing training patterns.
- Human evaluation. Programmers rating comments on usefulness, accuracy, and conciseness, on a Likert scale. A small N=50 study would say more than another decimal place of BLEU.